# 37. Chapter 3 원인판정과 타당성검토

Chapter 3 산출물을 모아 원인 가설별 지지/보류 상태를 정리합니다.

In [1]:
from pathlib import Path
import sys
import json
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

NOTEBOOK_DIR = Path.cwd()
if not (NOTEBOOK_DIR / "ch3_utils.py").exists():
    matches = list(Path.cwd().glob("Deeplearning/*/3장/ch3_utils.py")) + list(Path.cwd().glob("**/ch3_utils.py"))
    if matches:
        NOTEBOOK_DIR = matches[0].parent
    else:
        NOTEBOOK_DIR = Path("Deeplearning") / "Vision 응용" / "3장"
sys.path.insert(0, str(NOTEBOOK_DIR))

from ch3_utils import *

paths = find_ch3_paths()
set_korean_font()
set_seed(31)
paths

Chapter3Paths(chapter3_dir=WindowsPath('C:/Users/준승/Desktop/2026-1/Study/Deeplearning/Vision 응용/3장'), chapter2_2_dir=WindowsPath('C:/Users/준승/Desktop/2026-1/Study/Deeplearning/Vision 응용/2-2장'), data_root=WindowsPath('C:/Users/준승/Desktop/2026-1/Study/Deeplearning/Vision 응용/3장/data'), stress_ladder_root=WindowsPath('C:/Users/준승/Desktop/2026-1/Study/Deeplearning/Vision 응용/3장/data/synthetic_metal_stress_ladder'), runs_root=WindowsPath('C:/Users/준승/Desktop/2026-1/Study/Deeplearning/Vision 응용/3장/runs'), manifest_root=WindowsPath('C:/Users/준승/Desktop/2026-1/Study/Deeplearning/Vision 응용/3장/runs/manifests'), ch2_2_runs_root=WindowsPath('C:/Users/준승/Desktop/2026-1/Study/Deeplearning/Vision 응용/2-2장/runs'))

## 37-1. 산출물 존재 여부 확인

In [2]:
expected = {
    "photometric_counterfactual": paths.runs_root / "photometric_counterfactual" / "photometric_counterfactual_metrics.csv",
    "stage_feature_separability": paths.runs_root / "feature_separability" / "stage_feature_separability.csv",
    "scratch_token_coverage": paths.runs_root / "scratch_retention" / "scratch_token_coverage_with_metrics.csv",
    "architecture_ablation": paths.runs_root / "architecture_ablation" / "architecture_ablation_summary.csv",
    "stress_ladder": paths.runs_root / "stress_ladder" / "stress_ladder_metrics.csv",
}
status = pd.DataFrame(
    [{"artifact": name, "exists": path.exists(), "path": str(path)} for name, path in expected.items()]
)
display(status)

,artifact,exists,path
0,photometric_counterfactual,True,C:\Users\준승\Desktop\2026-1\Study\Deeplearning\...
1,stage_feature_separability,True,C:\Users\준승\Desktop\2026-1\Study\Deeplearning\...
2,scratch_token_coverage,True,C:\Users\준승\Desktop\2026-1\Study\Deeplearning\...
3,architecture_ablation,True,C:\Users\준승\Desktop\2026-1\Study\Deeplearning\...
4,stress_ladder,True,C:\Users\준승\Desktop\2026-1\Study\Deeplearning\...


## 37-2. 원인 판정 리포트 생성

In [3]:
report_path = build_chapter3_causal_report(paths.runs_root)
print(report_path)
print(report_path.read_text(encoding="utf-8")[:4000])

C:\Users\준승\Desktop\2026-1\Study\Deeplearning\Vision 응용\3장\runs\chapter3_causal_attribution_report.md
# Chapter 3 원인 판정 리포트

## 1. 산출물 존재 여부

| 산출물 | 상태 | 경로 |
|---|---|---|
| photometric_counterfactual | 통과 | `C:\Users\준승\Desktop\2026-1\Study\Deeplearning\Vision 응용\3장\runs\photometric_counterfactual\photometric_counterfactual_metrics.csv` |
| stage_feature_separability | 통과 | `C:\Users\준승\Desktop\2026-1\Study\Deeplearning\Vision 응용\3장\runs\feature_separability\stage_feature_separability.csv` |
| scratch_token_coverage | 통과 | `C:\Users\준승\Desktop\2026-1\Study\Deeplearning\Vision 응용\3장\runs\scratch_retention\scratch_token_coverage_with_metrics.csv` |
| architecture_ablation | 통과 | `C:\Users\준승\Desktop\2026-1\Study\Deeplearning\Vision 응용\3장\runs\architecture_ablation\architecture_ablation_summary.csv` |
| stress_ladder | 통과 | `C:\Users\준승\Desktop\2026-1\Study\Deeplearning\Vision 응용\3장\runs\stress_ladder\stress_ladder_metrics.csv` |

## 2. 자동 판정 초안

- H-A RGB shortcut: red original Dice=0

## 37-3. 타당성 검토 문서 생성

In [4]:
validity_lines = [
    "# Chapter 3 실험 타당성 검토",
    "",
    "## 내적 타당성",
    "- Chapter 2-2의 matched factorial manifest를 그대로 사용하므로 color/defect/shape 비교의 confounding 위험이 낮다.",
    "- Counterfactual 추론은 같은 mask와 shape에서 RGB appearance만 바꾸는 paired design이다.",
    "- 구조 ablation은 capacity, initialization, optimizer confounding이 있으므로 단독 결론으로 쓰지 않는다.",
    "",
    "## 통계 타당성",
    "- 최종 판정은 seed-level 반복을 우선한다.",
    "- sample-level bootstrap은 보조 지표로만 사용한다.",
    "- stress severe split은 바닥 효과가 있으므로 stress ladder 결과가 있어야 원인 판정이 가능하다.",
    "",
    "## 원인 판정 조건",
    "- H-A: photometric counterfactual 변화 + stage1/2 color separability + photometric_aug의 separability 감소.",
    "- H-B: scratch token coverage/activation retention 저하 + high-resolution/SR/stem ablation의 선택적 scratch 개선.",
    "- H-C: stress severity별 feature distance/FNR 증가 + no-glare 또는 roughness 완화에서 회복.",
    "- H-D: exposure 모델의 logit margin 이동 + balanced replay로 negative transfer 완화.",
]
validity_path = paths.runs_root / "chapter3_validity_review.md"
validity_path.write_text("\n".join(validity_lines), encoding="utf-8")
print(validity_path)

C:\Users\준승\Desktop\2026-1\Study\Deeplearning\Vision 응용\3장\runs\chapter3_validity_review.md
